# Free global elevation models in rich and poor coastal cities

This notebook reproduces the full analysis: ICESat-2 ATL08 ground heights are used as a common reference to measure the accuracy of three free global DEMs (Copernicus DEM GLO-30, FABDEM, DeltaDTM) in ten coastal cities, five in lower-middle-income and five in high-income countries. Airborne LiDAR is used in the high-income cities to test ICESat-2 itself.

**How to run:** open in Google Colab and run the cells from top to bottom. You need a NASA Earthdata account and a Google Earth Engine Cloud project (see README). Intermediate files are saved to Google Drive, so if Colab disconnects you can run the notebook again and finished steps load from Drive instead of repeating.

## 1. Setup
### 1.1 Install packages

In [ ]:
!pip install -q earthaccess earthengine-api h5py pyproj statsmodels

### 1.2 Connect Google Drive, Earth Engine and NASA Earthdata
Replace `YOUR-PROJECT-ID` with your own Earth Engine Cloud project ID. Earthdata asks for your username and password when this cell runs; they are not stored in the notebook.

In [ ]:
import os, glob, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from google.colab import drive

drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/T17'          # all outputs are saved here
os.makedirs(OUT, exist_ok=True)

import ee
ee.Authenticate()
ee.Initialize(project='YOUR-PROJECT-ID')

import earthaccess
earthaccess.login(persist=True)

### 1.3 Study cities
Each city is a rectangle (west, south, east, north) around its low-lying built-up coast. `group` is the World Bank income group (July 2026): `rich` = high income, `poor` = lower-middle income. `lidar` names the LiDAR source used to test ICESat-2.

In [ ]:
CITIES = {
    'lagos':         {'group': 'poor', 'bbox': (3.00, 6.38, 3.95, 6.75),         'lidar': None},
    'miami':         {'group': 'rich', 'bbox': (-80.50, 25.45, -80.08, 26.35),   'lidar': '3dep'},
    'mumbai':        {'group': 'poor', 'bbox': (72.77, 18.88, 73.05, 19.30),     'lidar': None},
    'new_york':      {'group': 'rich', 'bbox': (-74.27, 40.48, -73.70, 40.92),   'lidar': '3dep'},
    'karachi':       {'group': 'poor', 'bbox': (66.85, 24.75, 67.25, 25.05),     'lidar': None},
    'los_angeles':   {'group': 'rich', 'bbox': (-118.55, 33.70, -118.05, 34.10), 'lidar': '3dep'},
    'abidjan':       {'group': 'poor', 'bbox': (-4.15, 5.22, -3.85, 5.45),       'lidar': None},
    'houston':       {'group': 'rich', 'bbox': (-95.45, 29.25, -94.75, 29.85),   'lidar': '3dep'},
    'dar_es_salaam': {'group': 'poor', 'bbox': (39.15, -6.95, 39.40, -6.70),     'lidar': None},
    'london':        {'group': 'rich', 'bbox': (-0.25, 51.42, 0.35, 51.56),      'lidar': 'england'},
}
START, END = '2019-01-01', '2023-12-31'      # ICESat-2 period
N_POINTS = 50000                             # maximum points per city
SEED = 42                                    # makes the random sample repeatable

### 1.4 Helper functions
`error_stats` summarises a set of errors. `fetch_with_retry` downloads an Earth Engine table from its link and waits and retries if the server is busy.

In [ ]:
def error_stats(err):
    err = pd.Series(err).dropna()
    return pd.Series({
        'n': len(err),
        'bias_m': err.mean(),                                    # too high (+) or too low (-) on average
        'median_m': err.median(),
        'rmse_m': np.sqrt((err ** 2).mean()),
        'nmad_m': 1.4826 * (err - err.median()).abs().median(),  # robust typical error
        'le90_m': err.abs().quantile(0.9),                       # 90% of errors are smaller than this
    })

def fetch_with_retry(url, tries=5):
    for attempt in range(tries):
        try:
            return pd.read_csv(url)
        except Exception as e:
            wait = 30 * (attempt + 1)
            print(f'    server busy ({e}); trying again in {wait} s')
            time.sleep(wait)
    raise RuntimeError('Earth Engine kept failing. Wait a few minutes and run this cell again.')

def points_to_fc(df):
    return ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lo, la]), {'pid': p})
                                 for lo, la, p in zip(df.lon, df.lat, df.pid)])

## 2. Earth Engine layers and study-area check
All three DEMs use the EGM2008 geoid. DeltaDTM stores 30 for land above 30 m, so that value is masked. The low-lying urban area of each city is printed as a check.

In [ ]:
cop_col = ee.ImageCollection('COPERNICUS/DEM/GLO30_2024_1').select('DEM')
cop = cop_col.mosaic().setDefaultProjection(cop_col.first().projection()).rename('cop')
fab = ee.ImageCollection('projects/sat-io/open-datasets/FABDEM').mosaic().rename('fab')
delta = ee.Image('projects/sat-io/open-datasets/DELTARES/deltadtm_v1-1').select('b1')
delta = delta.updateMask(delta.neq(30)).rename('delta')

slope = ee.Terrain.slope(cop).rename('slope')
lc = ee.ImageCollection('ESA/WorldCover/v200').first().rename('lc')
smod = (ee.ImageCollection('JRC/GHSL/P2023A/GHS_SMOD_V2-0')
        .filterDate('2020-01-01', '2020-12-31').first()
        .select('smod_code').rename('smod'))

LIDAR = {
    '3dep': ee.ImageCollection('USGS/3DEP/1m').mosaic().rename('lidar'),                 # NAVD88
    'england': ee.Image('UK/EA/ENGLAND_1M_TERRAIN/2022').select('dtm').rename('lidar'),  # ODN
}

def area_km2(mask, region):
    a = ee.Image.pixelArea().updateMask(mask).reduceRegion(
        ee.Reducer.sum(), region, scale=30, maxPixels=1e10, tileScale=4).get('area')
    return ee.Number(a).divide(1e6).getInfo()

for name, c in CITIES.items():
    region = ee.Geometry.Rectangle(list(c['bbox']))
    km2 = area_km2(smod.gte(21).And(cop.lte(10)).And(lc.neq(80)), region)
    print(f'{name}: {km2:.0f} km2 of urban land below 10 m')

## 3. ICESat-2 reference points
### 3.1 Read one ATL08 file
Keeps strong beams only, uses the 20 m terrain sub-segments (falls back to 100 m segments in older files), removes fill values and keeps points inside the city rectangle.

In [ ]:
import h5py
BEAMS = ['gt1l', 'gt1r', 'gt2l', 'gt2r', 'gt3l', 'gt3r']

def read_atl08(path, bbox):
    w, s, e, n = bbox
    parts = []
    with h5py.File(path, 'r') as f:
        for b in BEAMS:
            if b not in f or 'land_segments' not in f[b]:
                continue
            kind = f[b].attrs.get('atlas_beam_type', b'')
            kind = kind.decode() if isinstance(kind, bytes) else str(kind)
            if kind != 'strong':
                continue
            ls = f[b]['land_segments']
            unc = ls['terrain']['h_te_uncertainty'][:]
            t = ls['delta_time'][:]
            if 'h_te_best_fit_20m' in ls['terrain']:
                lat = ls['latitude_20m'][:].ravel()
                lon = ls['longitude_20m'][:].ravel()
                h = ls['terrain']['h_te_best_fit_20m'][:].ravel()
                k = ls['latitude_20m'].shape[1]
                unc, t = np.repeat(unc, k), np.repeat(t, k)
            else:
                lat, lon = ls['latitude'][:], ls['longitude'][:]
                h = ls['terrain']['h_te_best_fit'][:]
            parts.append(pd.DataFrame({'lat': lat, 'lon': lon, 'h_ellip': h,
                                       'h_unc': unc, 'delta_time': t, 'beam': b,
                                       'file': os.path.basename(path)}))
    if not parts:
        return pd.DataFrame()
    df = pd.concat(parts, ignore_index=True)
    ok = (df.h_ellip.abs() < 1e4) & (df.h_unc.abs() < 1e4)       # removes fill values
    ok &= df.lon.between(w, e) & df.lat.between(s, n)
    return df[ok]

### 3.2 Download, read and delete ATL08 files
Files are downloaded ten at a time and deleted after reading, to save disk space. Only the newest data version is used. Results are saved per city.

In [ ]:
TMP = '/content/atl08_tmp'

def get_icesat2(name, bbox):
    out_csv = f'{OUT}/icesat2_raw_{name}.csv'
    if os.path.exists(out_csv):
        print(name, 'already downloaded')
        return pd.read_csv(out_csv)
    found = earthaccess.search_data(short_name='ATL08', bounding_box=bbox, temporal=(START, END))
    version = lambda g: g.data_links()[0].split('/')[-1].split('_')[3]
    newest = max(version(g) for g in found)
    found = [g for g in found if version(g) == newest]
    print(f'{name}: {len(found)} ATL08 files (version {newest})')
    parts = []
    for i in range(0, len(found), 10):
        paths = earthaccess.download(found[i:i + 10], local_path=TMP)
        for p in paths:
            parts.append(read_atl08(p, bbox))
            os.remove(p)
        print(f'  {min(i + 10, len(found))}/{len(found)} files read')
    df = pd.concat(parts, ignore_index=True)
    df.to_csv(out_csv, index=False)
    return df

raw = {name: get_icesat2(name, c['bbox']) for name, c in CITIES.items()}
for name, df in raw.items():
    print(f'{name}: {len(df):,} raw points')

### 3.3 Convert to EGM2008 and clean
Converts ellipsoidal heights to EGM2008, removes points with ATL08 terrain uncertainty of 3 m or more, keeps points between -5 and 10 m (selected by the reference height, not by any DEM), then draws a repeatable random sample of up to 50,000 points per city.

In [ ]:
import urllib.request, pyproj
from pyproj import Transformer

grid = os.path.join(pyproj.datadir.get_user_data_dir(True), 'us_nga_egm08_25.tif')
if not os.path.exists(grid):
    urllib.request.urlretrieve('https://cdn.proj.org/us_nga_egm08_25.tif', grid)
to_egm = Transformer.from_crs('EPSG:4979', 'EPSG:4326+3855', always_xy=True)

def clean_icesat2(name, df, max_unc=3.0):
    _, _, h = to_egm.transform(df.lon.values, df.lat.values, df.h_ellip.values)
    df = df.assign(h_egm=h)
    if not np.isfinite(df.h_egm).all():
        raise ValueError('EGM2008 conversion failed. Run this cell again.')
    n0 = len(df)
    df = df[df.h_unc < max_unc]
    df = df[df.h_egm.between(-5, 10)]
    df = df.reset_index(drop=True)
    df['pid'] = name + '_' + df.index.astype(str)
    gap = (df.h_ellip - df.h_egm).median()
    print(f'{name}: {n0} raw -> {len(df)} clean points. Ellipsoid-to-EGM2008 gap: {gap:.1f} m')
    df.to_csv(f'{OUT}/icesat2_clean_{name}.csv', index=False)
    return df

clean = {name: clean_icesat2(name, raw[name]) for name in CITIES}

for name in clean:
    if len(clean[name]) > N_POINTS:
        clean[name] = clean[name].sample(N_POINTS, random_state=SEED).reset_index(drop=True)
    print(f'{name}: {len(clean[name]):,} points kept')

## 4. Sample all layers at the ICESat-2 points
Reads the three DEMs, slope, land cover, urban class and (high-income cities) LiDAR at every point at 30 m, through Earth Engine download links, 2,000 points per request.

In [ ]:
def sample_city(name, pts, lidar_key, step=2000):
    out_csv = f'{OUT}/sampled_{name}.csv'
    if os.path.exists(out_csv):
        return pd.read_csv(out_csv)
    stack = cop.addBands([fab, delta, slope, lc, smod])
    if lidar_key:
        stack = stack.addBands(LIDAR[lidar_key])
    pieces = []
    for i in range(0, len(pts), step):
        sampled = stack.reduceRegions(collection=points_to_fc(pts.iloc[i:i + step]),
                                      reducer=ee.Reducer.first(), scale=30)
        pieces.append(fetch_with_retry(sampled.getDownloadURL(filetype='csv')))
        print(f'  {name}: {min(i + step, len(pts))}/{len(pts)} points sampled')
    s = pd.concat(pieces, ignore_index=True).drop(columns=['system:index', '.geo'], errors='ignore')
    df = pts.merge(s, on='pid', how='left')
    df.to_csv(out_csv, index=False)
    return df

frames = []
for name, c in CITIES.items():
    d = sample_city(name, clean[name], c['lidar'])
    d['city'], d['group'] = name, c['group']
    frames.append(d)

## 5. Join cities and filter
Keeps urban points (GHS-SMOD class 21 or more), removes water and rare land-cover classes, groups WorldCover into four classes, and removes points where Copernicus DEM and ICESat-2 differ by more than 50 m. A table of points remaining after each filter is saved for the methods.

In [ ]:
LC_NAMES = {10: 'Trees', 95: 'Trees', 20: 'Low vegetation', 30: 'Low vegetation',
            40: 'Low vegetation', 90: 'Low vegetation', 50: 'Built-up', 60: 'Bare'}

allp = pd.concat(frames, ignore_index=True)
urban = allp.smod >= 21
land_ok = urban & allp.lc.isin(list(LC_NAMES))
final = land_ok & ((allp.cop - allp.h_egm).abs() < 50)
filter_counts = (allp.assign(sampled=True, after_urban=urban, after_landcover=land_ok, after_50m=final)
                 .groupby('city')[['sampled', 'after_urban', 'after_landcover', 'after_50m']].sum())
filter_counts.to_csv(f'{OUT}/table_filter_counts.csv')
print(filter_counts)

data = allp[final].copy()
data['landcover'] = data.lc.map(LC_NAMES)
print(f'{len(allp)} points -> {len(data)} after filtering')
print(data.groupby(['city', 'landcover']).size().unstack(fill_value=0))
data.to_csv(f'{OUT}/all_points_clean.csv', index=False)

## 6. Test ICESat-2 against LiDAR (high-income cities)
The offset between each national LiDAR datum and EGM2008 is estimated as the median LiDAR minus ICESat-2 difference on open, flat land (low vegetation or bare, slope below 2 degrees) and removed. ICESat-2 is then compared with LiDAR, and each DEM is assessed against both references.

In [ ]:
rows, offsets = [], {}
for name, d in data[data.group == 'rich'].groupby('city'):
    d = d.dropna(subset=['lidar'])
    flat = d[d.lc.isin([30, 40, 60]) & (d.slope < 2)]
    if len(flat) < 100:
        flat = d[d.lc != 10]
    offsets[name] = (flat.lidar - flat.h_egm).median()
    data.loc[d.index, 'lidar_egm'] = d.lidar - offsets[name]
    for land, dd in [('All land', d), ('Open flat land', flat)]:
        rows.append({'city': name, 'land': land, 'zero_point_offset_m': offsets[name],
                     **error_stats(dd.h_egm - (dd.lidar - offsets[name]))})
icesat_check = pd.DataFrame(rows).round(2)
icesat_check.to_csv(f'{OUT}/table_icesat2_vs_lidar.csv', index=False)
print(icesat_check)

same = []
for name, d in data.dropna(subset=['lidar_egm']).groupby('city'):
    for dem in ['cop', 'fab', 'delta']:
        a, b = error_stats(d[dem] - d.h_egm), error_stats(d[dem] - d.lidar_egm)
        same.append({'city': name, 'dem': dem,
                     'nmad_vs_icesat2': a.nmad_m, 'nmad_vs_lidar': b.nmad_m,
                     'le90_vs_icesat2': a.le90_m, 'le90_vs_lidar': b.le90_m})
same = pd.DataFrame(same).round(2)
same.to_csv(f'{OUT}/table_icesat2_vs_lidar_dem_check.csv', index=False)
print(same)

## 7. DEM errors
Error = DEM height minus ICESat-2 height (positive = DEM too high). Summarised by city, and by land-cover group and income group.

In [ ]:
DEMS = ['cop', 'fab', 'delta']
long = data.melt(id_vars=['pid', 'city', 'group', 'landcover', 'slope', 'h_egm'],
                 value_vars=DEMS, var_name='dem', value_name='dem_h').dropna(subset=['dem_h'])
long['err'] = long.dem_h - long.h_egm

by_city = long.groupby(['dem', 'city']).err.apply(error_stats).unstack().round(2)
by_class = long.groupby(['dem', 'landcover', 'group']).err.apply(error_stats).unstack().round(2)
by_city.to_csv(f'{OUT}/table_errors_by_city.csv')
by_class.to_csv(f'{OUT}/table_errors_by_landcover_rich_poor.csv')
print(by_city)
print(by_class)

## 8. Mixed-effects models: lower-middle vs high income
Response: log(|error| + 0.05 m). Fixed effects: income group, land-cover group (all-land models) and slope. Random intercept: city. The poor/rich ratio is the exponent of the income coefficient.

In [ ]:
import statsmodels.formula.api as smf
warnings.filterwarnings('ignore')                    # convergence messages are common with ten cities

long['log_abs_err'] = np.log(long.err.abs() + 0.05)
K = "C(group, Treatment('rich'))[T.poor]"

def income_model(d, formula):
    m = smf.mixedlm(formula, d, groups=d['city']).fit()
    lo, hi = m.conf_int().loc[K]
    return m, {'poor_vs_rich': np.exp(m.params[K]), 'ci_low': np.exp(lo),
               'ci_high': np.exp(hi), 'p_value': m.pvalues[K],
               'city_spread': float(m.cov_re.iloc[0, 0])}

res_all, res_built = [], []
for dem, d in long.groupby('dem'):
    _, r = income_model(d, "log_abs_err ~ C(group, Treatment('rich')) + C(landcover) + slope")
    res_all.append({'dem': dem, **r})
    _, r = income_model(d[d.landcover == 'Built-up'], "log_abs_err ~ C(group, Treatment('rich')) + slope")
    res_built.append({'dem': dem, **r})

model_all = pd.DataFrame(res_all).round(3)
model_built = pd.DataFrame(res_built).round(3)
model_all.to_csv(f'{OUT}/table_model_poor_vs_rich.csv', index=False)
model_built.to_csv(f'{OUT}/table_model_poor_vs_rich_builtup.csv', index=False)
print('All land'); print(model_all)
print('Built-up land'); print(model_built)

## 9. Does building form explain the differences?
Samples GHSL building height (2018) and built-up fraction (2020) at 100 m, then refits the models with and without them.

In [ ]:
bh = ee.Image('JRC/GHSL/P2023A/GHS_BUILT_H/2018').select('built_height').rename('bldg_h')
bs = (ee.ImageCollection('JRC/GHSL/P2023A/GHS_BUILT_S')
      .filterDate('2020-01-01', '2020-12-31').first()
      .select('built_surface').divide(10000).rename('built_frac'))   # 0 = no buildings, 1 = fully built
form = bh.addBands(bs)

def sample_form(name, pts, step=2000):
    out_csv = f'{OUT}/form_{name}.csv'
    if os.path.exists(out_csv):
        return pd.read_csv(out_csv)
    pieces = []
    for i in range(0, len(pts), step):
        s = form.reduceRegions(collection=points_to_fc(pts.iloc[i:i + step]),
                               reducer=ee.Reducer.first(), scale=100)
        pieces.append(fetch_with_retry(s.getDownloadURL(filetype='csv')))
        print(f'  {name}: {min(i + step, len(pts))}/{len(pts)}')
    s = pd.concat(pieces, ignore_index=True).drop(columns=['system:index', '.geo'], errors='ignore')
    s.to_csv(out_csv, index=False)
    return s

form_all = pd.concat([sample_form(n, data[data.city == n][['pid', 'lat', 'lon']]) for n in CITIES],
                     ignore_index=True)
data = data.drop(columns=['bldg_h', 'built_frac'], errors='ignore').merge(form_all, on='pid', how='left')
print(data.groupby('city')[['bldg_h', 'built_frac']].median().round(2))

In [ ]:
long2 = long.merge(data[['pid', 'bldg_h', 'built_frac']], on='pid', how='left')
long2 = long2.dropna(subset=['bldg_h', 'built_frac'])

rows = []
for land, sub in [('All land', long2), ('Built-up', long2[long2.landcover == 'Built-up'])]:
    base = "log_abs_err ~ C(group, Treatment('rich')) + slope"
    if land == 'All land':
        base += ' + C(landcover)'
    for dem, d in sub.groupby('dem'):
        for model, formula in [('without building form', base),
                               ('with building form', base + ' + bldg_h + built_frac')]:
            m, r = income_model(d, formula)
            row = {'land': land, 'dem': dem, 'model': model, **r}
            if model == 'with building form':
                row['per_1m_taller'] = np.exp(m.params['bldg_h'])
                row['p_height'] = m.pvalues['bldg_h']
                row['per_10pct_denser'] = np.exp(0.1 * m.params['built_frac'])
                row['p_density'] = m.pvalues['built_frac']
            rows.append(row)

form_results = pd.DataFrame(rows).round(3)
form_results.to_csv(f'{OUT}/table_model_building_form.csv', index=False)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 20)
print(form_results)

## 10. Urban land below 1, 2 and 3 m
For each DEM (and offset-adjusted LiDAR in high-income cities), the urban, non-water area at or below each level above EGM2008. `uncertain_km2` is the area within one NMAD of each level for that DEM and city.

In [ ]:
flood_rows = []
for name, c in CITIES.items():
    region = ee.Geometry.Rectangle(list(c['bbox']))
    land = smod.gte(21).And(lc.neq(80))
    maps = {'cop': cop, 'fab': fab, 'delta': delta}
    if c['lidar']:
        maps['lidar'] = LIDAR[c['lidar']].subtract(offsets[name])
    for dem, img in maps.items():
        band = 0 if dem == 'lidar' else float(by_city.loc[(dem, name), 'nmad_m'])
        for level in [1, 2, 3]:
            flood_rows.append({
                'city': name, 'group': c['group'], 'dem': dem, 'below_m': level,
                'area_km2': area_km2(land.And(img.lte(level)), region),
                'uncertain_km2': area_km2(land.And(img.subtract(level).abs().lte(band)), region) if band else 0,
            })
    print(name, 'done')
flood = pd.DataFrame(flood_rows).round(1)
flood.to_csv(f'{OUT}/table_flood_area.csv', index=False)
print(flood)

## 11. Figures

In [ ]:
NAMES = {'cop': 'Copernicus DEM', 'fab': 'FABDEM', 'delta': 'DeltaDTM', 'lidar': 'LiDAR (reference)'}
GROUPS = {'rich': 'Rich cities', 'poor': 'Poor cities'}
CITY = {'lagos': 'Lagos', 'miami': 'Miami', 'mumbai': 'Mumbai', 'new_york': 'New York',
        'karachi': 'Karachi', 'los_angeles': 'Los Angeles', 'abidjan': 'Abidjan',
        'houston': 'Houston', 'dar_es_salaam': 'Dar es Salaam', 'london': 'London'}

# Figure 1: ICESat-2 against LiDAR, one panel per high-income city
r = data.dropna(subset=['lidar_egm'])
cities = sorted(r.city.unique())
fig, axes = plt.subplots(1, len(cities), figsize=(3 * len(cities), 3.4), sharex=True, sharey=True)
for ax, name in zip(axes, cities):
    d = r[r.city == name]
    ax.hexbin(d.lidar_egm, d.h_egm, gridsize=40, extent=(-3, 12, -3, 12),
              bins='log', cmap='viridis', mincnt=1)
    ax.plot([-3, 12], [-3, 12], color='red', lw=0.8)
    ax.text(-2.5, 10.8, f'NMAD = {error_stats(d.h_egm - d.lidar_egm).nmad_m:.2f} m', fontsize=9)
    ax.set_title(CITY[name])
    ax.set_xlim(-3, 12); ax.set_ylim(-3, 12); ax.set_aspect('equal')
    ax.set_xlabel('LiDAR height (m)')
axes[0].set_ylabel('ICESat-2 height (m)')
fig.savefig(f'{OUT}/fig1_icesat2_vs_lidar.png', dpi=300, bbox_inches='tight')

In [ ]:
# Figure 2: errors by land cover, rich vs poor, one panel per DEM
plot = long.assign(DEM=long.dem.map(NAMES), Group=long.group.map(GROUPS))
g = sns.catplot(data=plot, x='landcover', y='err', hue='Group', col='DEM', kind='box',
                showfliers=False, height=4, aspect=0.9,
                order=['Built-up', 'Low vegetation', 'Trees', 'Bare'],
                hue_order=['Rich cities', 'Poor cities'],
                col_order=['Copernicus DEM', 'FABDEM', 'DeltaDTM'],
                palette={'Rich cities': '#4C72B0', 'Poor cities': '#DD8452'})
g.set_titles('{col_name}')
g.set_axis_labels('', 'DEM minus ICESat-2 height (m)')
g._legend.set_title('')
for ax in g.axes.flat:
    ax.axhline(0, color='grey', lw=0.8)
    ax.tick_params(axis='x', rotation=30)
g.savefig(f'{OUT}/fig2_errors_rich_vs_poor.png', dpi=300, bbox_inches='tight')

In [ ]:
# Figure 3: urban land below 1, 2 and 3 m, cities shown in their pairs
PAIRS = [('lagos', 'miami'), ('mumbai', 'new_york'), ('karachi', 'los_angeles'),
         ('abidjan', 'houston'), ('dar_es_salaam', 'london')]
COLOURS = {'cop': '#4C72B0', 'fab': '#DD8452', 'delta': '#55A868', 'lidar': 'black'}

fig, axes = plt.subplots(5, 2, figsize=(9, 14))
for row, pair in enumerate(PAIRS):
    for col, name in enumerate(pair):
        ax = axes[row, col]
        d = flood[flood.city == name]
        for i, dem in enumerate(['cop', 'fab', 'delta', 'lidar']):
            dd = d[d.dem == dem]
            if dd.empty:
                continue
            ax.bar(dd.below_m + (i - 1.5) * 0.2, dd.area_km2, width=0.2,
                   color=COLOURS[dem], label=NAMES[dem])
        ax.set_xticks([1, 2, 3]); ax.set_xticklabels(['below 1 m', 'below 2 m', 'below 3 m'])
        ax.set_title(f"{CITY[name]} ({'poor' if col == 0 else 'rich'})")
        ax.set_ylabel('Urban area (km²)')
        ax.spines[['top', 'right']].set_visible(False)
handles, labels = axes[0, 1].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=4, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.97))
fig.savefig(f'{OUT}/fig3_flood_area.png', dpi=300, bbox_inches='tight')